References
- https://www.kaggle.com/code/richolson/ai-math-olympiad-qwen2-5-72b for showing how to submit
- https://www.kaggle.com/code/abdullahmeda/load-72b-awq-model-using-vllm-on-l4-x4
- https://www.kaggle.com/code/huikang/qwen2-5-math-1-5b-instruct
- https://www.kaggle.com/code/mpware/vllm-0-7 for the current installation script
- https://www.kaggle.com/code/richolson/ai-math-olympiad-qwen2-5-72b for showing how to submit
- https://www.kaggle.com/code/abdullahmeda/load-72b-awq-model-using-vllm-on-l4-x4

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
import re
import time
import random
import warnings
from collections import Counter
import numpy as np, pandas as pd, polars as pl

import torch
import vllm
from vllm import LLM, SamplingParams

import kaggle_evaluation.aimo_2_inference_server

warnings.simplefilter('ignore')
print('PyTorch version:', torch.__version__)
print('vLLM:', vllm.__version__)

INFO 03-20 16:11:48 __init__.py:183] Automatically detected platform cuda.
PyTorch version: 2.5.1+cu124
vLLM: 0.7.1


In [2]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True
seed_everything(seed=0)

start_time = time.time()
cutoff_time = start_time + (4 * 60 + 45) * 60
cutoff_times = [int(x) for x in np.linspace(cutoff_time, start_time + 60 * 60, 50 + 1)]

In [3]:
if os.getenv('KAGGLE_KERNEL_RUN_TYPE') or os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    #llm_model_pth = '/kaggle/input/m/huikang/deepseek-r1/transformers/deepseek-r1-distill-qwen-7b-awq-casperhansen/1'
    #llm_model_pth = '/kaggle/input/m/huikang/deepseek-r1/transformers/deepseek-r1-distill-qwen-14b-awq-casperhansen/1'
    #llm_model_pth = '/kaggle/input/m/huikang/deepseek-r1/transformers/deepseek-r1-distill-qwen-32b-awq-casperhansen/1'
    #llm_model_pth = '/kaggle/input/m/huikang/deepseek-r1/transformers/deepseek-aideepseek-r1-distill-qwen-14b-awq-neody/1'
    llm_model_pth = '/kaggle/input/qwen-14b-awq-casperhansen/transformers/default/1'
else:
    llm_model_pth = '/root/volume/KirillR/QwQ-32B-Preview-AWQ'

MAX_NUM_SEQS = 16 #32
MAX_MODEL_LEN = 8192 * 3 // 2 #12288 #8192 * 3 // 2 131072 

llm = LLM(
    llm_model_pth,
    dtype="half",                 # The data type for the model weights and activations
    max_num_seqs=MAX_NUM_SEQS,    # Maximum number of sequences per iteration. Default is 256
    max_model_len=MAX_MODEL_LEN,  # Model context length
    trust_remote_code=True,       # Trust remote code (e.g., from HuggingFace) when downloading the model and tokenizer
    tensor_parallel_size=4,       # The number of GPUs to use for distributed execution with tensor parallelism
    gpu_memory_utilization=0.95,  # The ratio (between 0 and 1) of GPU memory to reserve for the model
    seed=2024,
)

tokenizer = llm.get_tokenizer()

WARNING 03-20 16:11:54 config.py:2368] Casting torch.bfloat16 to torch.float16.
INFO 03-20 16:12:22 config.py:526] This model supports multiple tasks: {'reward', 'score', 'classify', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 03-20 16:12:26 awq_marlin.py:109] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 03-20 16:12:26 config.py:1383] Defaulting to use mp for distributed inference
WARNING 03-20 16:12:26 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
INFO 03-20 16:12:26 llm_engine.py:232] Initializing a V0 LLM engine (v0.7.1) with config: model='/kaggle/input/qwen-14b-awq-casperhansen/transformers/default/1', speculative_config=None, tokenizer='/kaggle/input/qwen-14b-awq-casperhansen/transformers/default/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=12288, download_dir=None, 

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(VllmWorkerProcess pid=359) INFO 03-20 16:13:45 model_runner.py:1116] Loading model weights took 2.3731 GB
INFO 03-20 16:13:45 model_runner.py:1116] Loading model weights took 2.3731 GB
(VllmWorkerProcess pid=354) INFO 03-20 16:13:45 model_runner.py:1116] Loading model weights took 2.3731 GB
(VllmWorkerProcess pid=351) INFO 03-20 16:13:45 model_runner.py:1116] Loading model weights took 2.3731 GB
(VllmWorkerProcess pid=359) (VllmWorkerProcess pid=351) WARNING 03-20 16:14:05 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
WARNING 03-20 16:14:05 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
(VllmWorkerProcess pid=354) (VllmWorkerProcess pid=359) (VllmWorkerProcess pid=351) WARNING 03-20 16:14:05 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
WARNING 03-20 16:14:05 config.py:975] MLA is not supported with awq_marlin quantization. Disabling MLA.
WARNING 03-20 16:14:05 config.py:975] MLA is

Capturing CUDA graph shapes: 100%|██████████| 5/5 [00:05<00:00,  1.11s/it]

INFO 03-20 16:14:16 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
(VllmWorkerProcess pid=359) (VllmWorkerProcess pid=354) INFO 03-20 16:14:16 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
(VllmWorkerProcess pid=351) INFO 03-20 16:14:16 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
INFO 03-20 16:14:16 model_runner.py:1563] Graph capturing finished in 6 secs, took 0.09 GiB
INFO 03-20 16:14:16 llm_engine.py:429] init engine (profile, create kv cache, warmup model) took 31.33 seconds


In [4]:
def extract_boxed_text(text):
    pattern = r'oxed{(.*?)}'
    matches = re.findall(pattern, text)
    if not matches:
        return ""
    for match in matches[::-1]:
        if match != "":
            return match
    return ""

def batch_message_filter(list_of_messages) -> tuple[list[list[dict]], list[str]]:
    extracted_answers = []
    list_of_messages_to_keep = []
    for messages in list_of_messages:
        answer = extract_boxed_text(messages[-1]['content'])
        if answer:
            extracted_answers.append(answer)
        else:
            list_of_messages_to_keep.append(messages)
    return list_of_messages_to_keep, extracted_answers

def select_answer(answers):
    counter = Counter()
    for answer in answers:
        try:
            if int(answer) == float(answer):
                counter[int(answer)] += 1 + random.random() / 1_000
        except:
            pass
    if not counter:
        return 210
    _, answer = sorted([(v,k) for k,v in counter.items()], reverse=True)[0]
    return answer%1000

def batch_message_generate(list_of_messages) -> list[list[dict]]:
    max_tokens = MAX_MODEL_LEN
    if time.time() > cutoff_times[-1]:
        print("Speedrun")
        max_tokens = 2 * MAX_MODEL_LEN // 3

    sampling_params = SamplingParams(
        temperature=1.0,               # Randomness of the sampling
        top_p=0.90,                    # Cumulative probability of the top tokens to consider
        min_p=0.05,                    # Minimum probability for a token to be considered
        skip_special_tokens=True,      # Whether to skip special tokens in the output
        max_tokens=max_tokens,         # Maximum number of tokens to generate
        stop=["</think>"],             # List of strings that stop the generation
        seed=777,
    )
    
    list_of_texts = [
        tokenizer.apply_chat_template(
            conversation=messages,
            tokenize=False,
            add_generation_prompt=True
        )
        for messages in list_of_messages
    ]

    request_output = llm.generate(
        prompts=list_of_texts,
        sampling_params=sampling_params,
    )
    print([len(single_request_output.outputs[0].token_ids) for single_request_output in request_output])

    sort_keys_and_list_of_messages = []
    for messages, single_request_output in zip(list_of_messages, request_output):
        #print()
        #print(single_request_output.outputs[0].text)
        #print()
        messages.append({'role': 'assistant', 'content': single_request_output.outputs[0].text})

        sort_keys_and_list_of_messages.append(
            (
                len(single_request_output.outputs[0].token_ids),
                messages
            )
        )
    print([sort_key for sort_key, _ in sort_keys_and_list_of_messages])
    sort_keys_and_list_of_messages.sort(key=lambda sort_key_and_messages: sort_key_and_messages[0])
    print([sort_key for sort_key, _ in sort_keys_and_list_of_messages])
    
    list_of_messages = [messages for _, messages in sort_keys_and_list_of_messages]
    return list_of_messages

In [5]:
def create_starter_messages(question, index):
    options = []
    for _ in range(13):
        options.append(
            [
                {"role": "system", "content": "You are a helpful and harmless assistant. You are Qwen developed by Alibaba. You should think step-by-step. Return final answer within \\boxed{}, after taking modulo 1000."},
                {"role": "user", "content": question},
            ]
        )
    for _ in range(3):    
        options.append(
            [
                {"role": "system", "content": "You are a helpful and harmless assistant. You are Qwen developed by Alibaba. You should think step-by-step. After you get your final answer, take modulo 1000, and return the final answer within \\boxed{}."},
                {"role": "user", "content": question},
            ],
        )
    return options[index%len(options)]

def predict_for_question(question: str) -> int:
    selected_questions_only = True
    #selected_questions_only = False
    if selected_questions_only and not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
        #if "Triangle" not in question:
        #    return 210
        if "Triangle" not in question and "delightful" not in question and "George" not in question:
            return 210

    if time.time() > cutoff_time:
        return 210
    
    print(question)

    num_seqs = MAX_NUM_SEQS
    if time.time() > cutoff_times[-1]:
        num_seqs = 2 * MAX_NUM_SEQS // 3
    
    list_of_messages = [create_starter_messages(question, index) for index in range(num_seqs)]

    all_extracted_answers = []
    for _ in range(1):
        list_of_messages = batch_message_generate(list_of_messages)
        
        if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
            df = pd.DataFrame(
                {
                    "question": [question] * len(list_of_messages),
                    "message": [messages[-1]["content"] for messages in list_of_messages],
                }
            )
            df.to_csv(f"{str(int(time.time() - start_time)).zfill(5)}.csv", index=False)
        
        list_of_messages, extracted_answers = batch_message_filter(list_of_messages)
        all_extracted_answers.extend(extracted_answers)
    
    print(all_extracted_answers)
    answer = select_answer(all_extracted_answers)
    print(answer)

    print("\n\n")
    cutoff_times.pop()
    return answer

def predict(id_: pl.DataFrame, question: pl.DataFrame) -> pl.DataFrame | pd.DataFrame:
    id_ = id_.item(0)
    print("------")
    print(id_)
    question = question.item(0)
    answer = predict_for_question(question)
    print(question)
    print("------\n\n")
    return pl.DataFrame({'id': id_, 'answer': answer})

In [6]:
#predict_for_question("Triangle $ABC$ has side length $AB = 120$ and circumradius $R = 100$. Let $D$ be the foot of the perpendicular from $C$ to the line $AB$. What is the greatest possible length of segment $CD$?")

In [7]:
pd.read_csv(
    '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/reference.csv'
).drop('answer', axis=1).to_csv('reference.csv', index=False)

In [8]:
inference_server = kaggle_evaluation.aimo_2_inference_server.AIMO2InferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        (
#            '/kaggle/input/ai-mathematical-olympiad-progress-prize-2/test.csv',
            'reference.csv',
        )
    )

------
a1d40b
The Fibonacci numbers are defined as follows: $F_0 = 0$, $F_1 = 1$, and $F_{n+1} = F_n + F_{n-1}$ for $n \geq 1$. There are $N$ positive integers $n$ strictly less than $10^{101}$ such that $n^2 + (n+1)^2$ is a multiple of 5 but $F_{n-1}^2 + F_n^2$ is not. How many prime factors does $N$ have, counted with multiplicity?
------


------
192e23
Fred and George take part in a tennis tournament with $4046$ other players. In each round, the players are paired into $2024$ matches. How many ways are there to arrange the first round such that Fred and George do not have to play each other? (Two arrangements for the first round are \textit{different} if there is a player with a different opponent in the two arrangements.)


Processed prompts: 100%|██████████| 16/16 [08:20<00:00, 31.28s/it, est. speed input: 4.20 toks/s, output: 291.51 toks/s]


[5967, 8588, 9784, 12159, 11300, 8844, 7614, 8511, 10291, 11241, 12159, 7763, 7852, 8581, 7091, 8134]
[5967, 8588, 9784, 12159, 11300, 8844, 7614, 8511, 10291, 11241, 12159, 7763, 7852, 8581, 7091, 8134]
[5967, 7091, 7614, 7763, 7852, 8134, 8511, 8581, 8588, 8844, 9784, 10291, 11241, 11300, 12159, 12159]
['250', '250', '250', '250', '250', '250', '0', '0', '750', '0', '250', '250', '750', '250']
250



Fred and George take part in a tennis tournament with $4046$ other players. In each round, the players are paired into $2024$ matches. How many ways are there to arrange the first round such that Fred and George do not have to play each other? (Two arrangements for the first round are \textit{different} if there is a player with a different opponent in the two arrangements.)
------


------
88c219
For positive integers $x_1,\ldots, x_n$ define $G(x_1, \ldots, x_n)$ to be the sum of their $\frac{n(n-1)}{2}$ pairwise greatest common divisors. We say that an integer $n \geq 2$ is \emph{arti

Processed prompts: 100%|██████████| 16/16 [07:08<00:00, 26.79s/it, est. speed input: 3.83 toks/s, output: 236.27 toks/s]


[3070, 4916, 4643, 3261, 5788, 5544, 4750, 11984, 11105, 8953, 8174, 4386, 5392, 4436, 6226, 8662]
[3070, 4916, 4643, 3261, 5788, 5544, 4750, 11984, 11105, 8953, 8174, 4386, 5392, 4436, 6226, 8662]
[3070, 3261, 4386, 4436, 4643, 4750, 4916, 5392, 5544, 5788, 6226, 8174, 8662, 8953, 11105, 11984]
['180', '180', '180', '180', '180', '180', '180', '180', '180', '180', '180', '180', '180', '180', '180', '180']
180



Triangle $ABC$ has side length $AB = 120$ and circumradius $R = 100$. Let $D$ be the foot of the perpendicular from $C$ to the line $AB$. What is the greatest possible length of segment $CD$?
------


------
349493
We call a sequence $a_1, a_2, \ldots$ of non-negative integers \textit{delightful} if there exists a positive integer $N$ such that for all $n > N$, $a_n = 0$, and for all $i \geq 1$, $a_i$ counts the number of multiples of $i$ in $a_1, a_2, \ldots, a_N$. How many delightful sequences of non-negative integers are there?


Processed prompts: 100%|██████████| 16/16 [09:15<00:00, 34.70s/it, est. speed input: 4.34 toks/s, output: 348.36 toks/s]

[12140, 12140, 12140, 12140, 12140, 12140, 12041, 12140, 11430, 12140, 12140, 12140, 12140, 12132, 12132, 12132]
[12140, 12140, 12140, 12140, 12140, 12140, 12041, 12140, 11430, 12140, 12140, 12140, 12140, 12132, 12132, 12132]
[11430, 12041, 12132, 12132, 12132, 12140, 12140, 12140, 12140, 12140, 12140, 12140, 12140, 12140, 12140, 12140]
['2', '3']
3



We call a sequence $a_1, a_2, \ldots$ of non-negative integers \textit{delightful} if there exists a positive integer $N$ such that for all $n > N$, $a_n = 0$, and for all $i \geq 1$, $a_i$ counts the number of multiples of $i$ in $a_1, a_2, \ldots, a_N$. How many delightful sequences of non-negative integers are there?
------


------
057f8a
Three airline companies operate flights from Dodola island. Each company has a different schedule of departures. The first company departs every 100 days, the second every 120 days and the third every 150 days. What is the greatest positive integer $d$ for which it is true that there will be $d$ con